# DocuMind — Phase 2C: LayoutLMv3 SROIE Fine-Tuning & Benchmarking (GPU)

This notebook runs end-to-end training, held-out test evaluation, and scientific baseline comparison using **real SROIE receipt images** on Google Colab.

**Runtime:** Select *Runtime -> Change runtime type -> T4 GPU* before running.

In [ ]:
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq tesseract-ocr
!pip install -q torch torchvision transformers datasets seqeval accelerate pymupdf pytesseract opencv-python-headless pillow pydantic

In [ ]:
import os
if not os.path.exists('DocuMind'):
    !git clone https://github.com/Abhiram-C-Divakaran/DocuMind.git
%cd DocuMind
!git status -sb

## 1. Prepare Real SROIE Dataset Splits

Downloads `arvindrajan92/sroie_document_understanding` from Hugging Face Hub, persists real RGB receipt images, checks SHA-256 image hashes across splits to prevent leakage, aligns BIO entities (`COMPANY`, `DATE`, `ADDRESS`, `TOTAL`), and writes `data/sroie_processed/`.

In [ ]:
!python -m ml.data.prepare_dataset \
  --hf-dataset arvindrajan92/sroie_document_understanding \
  --output-dir data/sroie_processed \
  --seed 42

with open('data/sroie_processed/validation_report.md') as f:
    print(f.read())

## 2. Fine-Tune `microsoft/layoutlmv3-base`

Uses CUDA AMP (`fp16`), gradient accumulation, linear warmup, validation after every epoch, early stopping, and best-checkpoint selection on validation F1.

In [ ]:
!python -m ml.training.train_layoutlmv3 \
  --data-dir data/sroie_processed \
  --output-dir checkpoints/layoutlmv3-sroie \
  --best-model-dir checkpoints/layoutlmv3-sroie-best \
  --model-name microsoft/layoutlmv3-base \
  --epochs 8 \
  --batch-size 4 \
  --grad-accum 2 \
  --lr 2e-5 \
  --patience 3 \
  --seed 42

## 3. Held-Out Test Evaluation & Baseline Benchmark

In [ ]:
!python -m ml.evaluation.evaluate \
  --checkpoint-dir checkpoints/layoutlmv3-sroie-best \
  --data-dir data/sroie_processed \
  --split test \
  --output-dir ml/evaluation/results

!python -m ml.evaluation.compare_models \
  --checkpoint-dir checkpoints/layoutlmv3-sroie-best \
  --data-dir data/sroie_processed \
  --split test \
  --output-dir ml/evaluation/results \
  --run-full-ocr

with open('ml/evaluation/results/benchmark_comparison.md') as f:
    print(f.read())

## 4. Export Best Checkpoint & Reports

Archives `checkpoints/layoutlmv3-sroie-best/` and `ml/evaluation/results/` for local FastAPI serving (`LAYOUTLMV3_CHECKPOINT_DIR=checkpoints/layoutlmv3-sroie-best`).

In [ ]:
!zip -r -q documind_layoutlmv3_best.zip checkpoints/layoutlmv3-sroie-best ml/evaluation/results/evaluation_report.* ml/evaluation/results/benchmark_comparison.*
try:
    from google.colab import files
    files.download('documind_layoutlmv3_best.zip')
except ImportError:
    print('Saved archive at documind_layoutlmv3_best.zip')